# 🤖 Pipeline RAG — Question-Answering sur la Documentation Technique

## Rapport de Stage — INSEA Rabat, Filière Data Science

**Organisme d'accueil :** 3D Smart Factory  
**Année universitaire :** 2024–2025

---

### Description du projet

Ce notebook implémente un système complet de **Retrieval-Augmented Generation (RAG)** pour le question-answering sur la documentation technique de :
- **Python 3.13** (documentation officielle)
- **Scikit-learn 1.5** (documentation officielle)
- **LangChain** (documentation officielle)

### Architecture du pipeline (6 étapes)

| Étape | Description | Référence |
|:---:|---|---|
| 1 | **Collecte** — Clonage des dépôts GitHub officiels | — |
| 2 | **Nettoyage** — Parsing RST/MDX, normalisation | — |
| 3 | **Benchmarking** — Grid search 108 configurations | Gao et al. (2024) |
| 4 | **Indexation** — FAISS avec config optimale | Johnson et al. (2019) |
| 5 | **Génération** — RAG avec Mistral 7B Instruct | Lewis (2020), Jiang (2023) |
| 6 | **Évaluation** — Métriques RAGAS | Es et al. (2024) |

### Références bibliographiques principales

- Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS.
- Vaswani, A. et al. (2017). *Attention Is All You Need*. NeurIPS.
- Reimers, N. & Gurevych, I. (2019). *Sentence-BERT*. EMNLP.
- Gao, Y. et al. (2024). *Retrieval-Augmented Generation for Large Language Models: A Survey*.
- Es, S. et al. (2024). *RAGAS: Automated Evaluation of Retrieval Augmented Generation*.
- Jiang, A.Q. et al. (2023). *Mistral 7B*. arXiv:2310.06825.

---

## ⚙️ 0. Configuration de l'environnement

### Prérequis Kaggle
- **GPU** : Activer le GPU T4 (16 Go VRAM) dans *Settings → Accelerator → GPU T4 x2*
- **Internet** : Activer l'accès Internet dans *Settings → Internet → On*

### Instructions de téléversement du projet
1. Compresser le dossier `rag_project_src/` en `.zip`
2. Dans Kaggle : *Add Data → Upload → New Dataset* → téléverser le `.zip`
3. Le projet sera disponible dans `/kaggle/input/rag-pipeline-src/`

> **Alternative** : Si le projet est sur GitHub, on clone directement (cellule ci-dessous).

In [ ]:
# ============================================================
# 0.1 — Vérification du GPU
# ============================================================
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"✅ GPU détecté : {gpu_name} ({gpu_mem:.1f} Go VRAM)")
else:
    print("⚠️  Aucun GPU détecté ! Activez le GPU dans Settings → Accelerator.")
    print("    Le pipeline fonctionnera en mode CPU (très lent pour l'étape 5).")

In [ ]:
# ============================================================
# 0.2 — Installation des dépendances
# ============================================================
# Les bibliothèques de base (torch, transformers) sont pré-installées
# sur Kaggle. On installe uniquement les manquantes.

!pip install -q gitpython ftfy tiktoken sentence-transformers \
    faiss-cpu bitsandbytes accelerate rank_bm25 2>&1 | tail -5

print("\n✅ Toutes les dépendances sont installées.")

In [ ]:
# ============================================================
# 0.3 — Copie du projet dans le répertoire de travail
# ============================================================
import os
import shutil

# Adapter ce chemin selon votre méthode de téléversement :
#   - Dataset Kaggle : "/kaggle/input/rag-pipeline-src/rag_project_src"
#   - Upload direct  : "/kaggle/input/rag-project-src"

INPUT_DIR = "/kaggle/input"  # Dossier racine des datasets Kaggle
WORK_DIR = "/kaggle/working/rag_project_src"

# Chercher automatiquement le dossier du projet
project_found = False
for root, dirs, files in os.walk(INPUT_DIR):
    if "config.py" in files and "etape1_collecte.py" in files:
        src = root
        project_found = True
        break

if project_found:
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    shutil.copytree(src, WORK_DIR)
    print(f"✅ Projet copié depuis {src} → {WORK_DIR}")
else:
    print("⚠️  Projet non trouvé dans /kaggle/input/.")
    print("    Veuillez téléverser le dossier rag_project_src comme dataset.")
    print("    Ou décommentez la cellule GitHub ci-dessous.")

In [ ]:
# ============================================================
# 0.3 bis — Alternative : cloner depuis GitHub (si disponible)
# ============================================================
# Décommentez ces lignes si votre projet est sur GitHub :

# GITHUB_URL = "https://github.com/VOTRE_NOM/rag-pipeline.git"
# !git clone {GITHUB_URL} /kaggle/working/rag_project_src
# WORK_DIR = "/kaggle/working/rag_project_src"
# print(f"✅ Projet cloné depuis GitHub → {WORK_DIR}")

In [ ]:
# ============================================================
# 0.4 — Configuration du path Python
# ============================================================
import sys

WORK_DIR = "/kaggle/working/rag_project_src"

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

os.chdir(WORK_DIR)
print(f"📂 Répertoire de travail : {os.getcwd()}")

# Vérifier que les modules sont accessibles
from config import PROJECT_ROOT, DATA_DIR, BENCHMARK_DIR, VECTORSTORE_DIR, EVALUATION_DIR, LLM_CONFIG, init_directories
init_directories()

print(f"📁 Racine du projet : {PROJECT_ROOT}")
print(f"📁 Données : {DATA_DIR}")
print(f"🧠 LLM configuré : {LLM_CONFIG['model_name']}")
print(f"✅ Configuration chargée avec succès.")

---

## 📥 Étape 1 — Collecte documentaire

### Objectif
Collecter la documentation technique officielle de **Python 3.13**, **Scikit-learn 1.5** et **LangChain** à partir de leurs dépôts GitHub.

### Méthode
La collecte s'effectue par **clonage partiel** (*sparse checkout*) des dépôts Git officiels :
- `github.com/python/cpython` → dossier `Doc/` (fichiers `.rst`)
- `github.com/scikit-learn/scikit-learn` → dossier `doc/` (fichiers `.rst`)
- `github.com/langchain-ai/docs` → dossiers ciblés (fichiers `.mdx`)

Cette approche est préférable au *web scraping* car elle garantit la **reproductibilité** (version exacte du dépôt) et la **complétude** du corpus.

### Sortie attendue
- `data/raw/{python,sklearn,langchain}_docs/json_docs/*.json`
- ~1 200 documents bruts au total

> ⏱️ **Durée estimée** : 5–15 minutes (dépend de la connexion Internet)

In [ ]:
# ============================================================
# ÉTAPE 1 — COLLECTE DES DONNÉES
# ============================================================
from etape1_collecte import main as run_etape1

run_etape1()

In [ ]:
# Vérification du volume collecté
from config import PYTHON_RAW, SKLEARN_RAW, LANGCHAIN_RAW

for name, path in [("Python", PYTHON_RAW), ("Sklearn", SKLEARN_RAW), ("LangChain", LANGCHAIN_RAW)]:
    json_dir = path / "json_docs"
    if json_dir.exists():
        count = len(list(json_dir.glob("*.json")))
        print(f"  {name:12s} : {count} documents")
    else:
        print(f"  {name:12s} : ⚠️ Dossier non trouvé")

---

## 🧹 Étape 2 — Nettoyage documentaire

### Objectif
Normaliser et épurer les documents bruts pour obtenir un corpus de haute qualité, adapté au découpage et à la vectorisation.

### Opérations de nettoyage
- **Parsing structurel** : extraction des titres, sections et paragraphes depuis RST et MDX
- **Suppression du bruit** : directives Sphinx (`.. toctree::`, `.. note::`), frontmatter YAML, imports JSX
- **Normalisation** : espaces multiples, sauts de ligne, caractères Unicode (via `ftfy`)
- **Préservation du code** : les blocs de code sont balisés avec `[CODE: lang]` pour maintenir leur intégrité sémantique
- **Filtrage** : suppression des documents vides ou trop courts (< seuil minimal)

### Sortie attendue
- `data/processed/cleaned/{python,sklearn,langchain}/*.json`
- ~1 050 documents nettoyés (réduction ~19%)

> ⏱️ **Durée estimée** : 2–5 minutes

In [ ]:
# ============================================================
# ÉTAPE 2 — NETTOYAGE ET PRÉPARATION
# ============================================================
from etape2_nettoyage import main as run_etape2

run_etape2()

In [ ]:
# Vérification : nombre de documents nettoyés par source
from config import CLEAN_DIR
import json

total = 0
for source in ["python", "sklearn", "langchain"]:
    src_dir = CLEAN_DIR / source
    if src_dir.exists():
        files = list(src_dir.glob("*.json"))
        total += len(files)
        # Exemple : afficher le premier document
        if files:
            with open(files[0], 'r', encoding='utf-8') as f:
                doc = json.load(f)
            print(f"  {source:12s} : {len(files):>4} docs  │  Exemple : {doc.get('title', 'N/A')[:60]}")
    else:
        print(f"  {source:12s} : ⚠️ Non trouvé")

print(f"\n  Total : {total} documents nettoyés")

---

## 📊 Étape 3 — Benchmarking (Grid Search 108 configurations)

### Objectif
Identifier empiriquement la **combinaison optimale** de paramètres du pipeline par une évaluation exhaustive, conformément aux recommandations de Gao et al. (2024).

### Espace de recherche (produit cartésien)

| Dimension | Valeurs testées | Nombre |
|---|---|:---:|
| **Taille de chunk** | 256, 400, 512 tokens | 3 |
| **Chevauchement** | 30, 50, 80 tokens | 3 |
| **Modèle d'embedding** | `all-MiniLM-L6-v2`, `paraphrase-multilingual-MiniLM-L12-v2`, `all-mpnet-base-v2` | 3 |
| **Méthode de recherche** | Sémantique pure, BM25 pure, Hybride α=0.7, Hybride α=0.5 | 4 |

**Total : 3 × 3 × 3 × 4 = 108 configurations**

### Métriques d'évaluation (k=5)
- **Hit Rate@5** : proportion de requêtes avec ≥1 résultat pertinent dans le top-5
- **MRR@5** : Mean Reciprocal Rank — position moyenne du premier résultat pertinent
- **Precision@5** : proportion de résultats pertinents dans le top-5

### Jeu de test
50 questions techniques (factuelles, conceptuelles, procédurales) couvrant les 3 sources.

### Sortie attendue
- `data/benchmarks/benchmark_full_grid.csv` — résultats des 108 configurations
- `data/benchmarks/benchmark_report.json` — configuration optimale recommandée

> ⏱️ **Durée estimée** : 2–4 heures (GPU accélère les embeddings)
>
> 💡 **Optimisation** : les embeddings sont pré-calculés par paire (chunk_config, modèle) → 27 passes coûteuses + 108 évaluations légères.

In [ ]:
# ============================================================
# ÉTAPE 3 — BENCHMARKING (GRID SEARCH 108 CONFIGURATIONS)
# ============================================================
from etape3_benchmarking import main as run_etape3

run_etape3()

In [ ]:
# ============================================================
# Analyse des résultats du benchmarking
# ============================================================
import pandas as pd
from config import BENCHMARK_DIR

# Charger les résultats du grid search
grid_path = BENCHMARK_DIR / "benchmark_full_grid.csv"
if grid_path.exists():
    df = pd.read_csv(grid_path)
    print(f"📊 {len(df)} configurations évaluées\n")
    
    # Top 10 configurations par MRR@5
    print("🏆 Top 10 configurations (par MRR@5) :")
    print("─" * 90)
    top10 = df.nlargest(10, 'mrr@5')[['chunk_size', 'overlap', 'model', 'search_method', 'mrr@5', 'hit_rate@5', 'precision@5']]
    print(top10.to_string(index=False))
    
    # Configuration optimale
    best = df.loc[df['mrr@5'].idxmax()]
    print(f"\n{'='*60}")
    print(f"🥇 CONFIGURATION OPTIMALE :")
    print(f"   Chunk size  : {int(best['chunk_size'])} tokens")
    print(f"   Overlap     : {int(best['overlap'])} tokens")
    print(f"   Embedding   : {best['model']}")
    print(f"   Recherche   : {best['search_method']}")
    print(f"   MRR@5       : {best['mrr@5']:.4f}")
    print(f"   Hit Rate@5  : {best['hit_rate@5']:.4f}")
    print(f"   Precision@5 : {best['precision@5']:.4f}")
    print(f"{'='*60}")
else:
    print("⚠️ Fichier benchmark_full_grid.csv non trouvé. Exécutez l'étape 3.")

In [ ]:
# ============================================================
# Visualisation des résultats du benchmarking
# ============================================================
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['font.size'] = 11

if grid_path.exists():
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('Résultats du Benchmarking — Grid Search 108 configurations', fontsize=14, fontweight='bold')
    
    # 1. MRR par modèle d'embedding
    ax1 = axes[0]
    for model in df['model'].unique():
        subset = df[df['model'] == model]
        ax1.bar(model.split('/')[-1][:15], subset['mrr@5'].mean(), 
                yerr=subset['mrr@5'].std(), capsize=5, alpha=0.8)
    ax1.set_title('MRR@5 moyen par modèle d\'embedding')
    ax1.set_ylabel('MRR@5')
    ax1.tick_params(axis='x', rotation=20)
    
    # 2. MRR par méthode de recherche
    ax2 = axes[1]
    for method in df['search_method'].unique():
        subset = df[df['search_method'] == method]
        ax2.bar(method[:18], subset['mrr@5'].mean(),
                yerr=subset['mrr@5'].std(), capsize=5, alpha=0.8)
    ax2.set_title('MRR@5 moyen par méthode de recherche')
    ax2.set_ylabel('MRR@5')
    ax2.tick_params(axis='x', rotation=20)
    
    # 3. MRR par taille de chunk
    ax3 = axes[2]
    for cs in sorted(df['chunk_size'].unique()):
        subset = df[df['chunk_size'] == cs]
        ax3.bar(str(int(cs)), subset['mrr@5'].mean(),
                yerr=subset['mrr@5'].std(), capsize=5, alpha=0.8)
    ax3.set_title('MRR@5 moyen par taille de chunk')
    ax3.set_ylabel('MRR@5')
    ax3.set_xlabel('Chunk size (tokens)')
    
    plt.tight_layout()
    plt.savefig(BENCHMARK_DIR / 'benchmark_visualisation.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n📊 Graphique sauvegardé : {BENCHMARK_DIR / 'benchmark_visualisation.png'}")

In [ ]:
# ============================================================
# Heatmap : interaction chunk_size × modèle d'embedding
# ============================================================
import seaborn as sns

if grid_path.exists():
    # MRR@5 moyen par (chunk_size, modèle) — agrégé sur les 4 méthodes de recherche
    pivot = df.pivot_table(
        values='mrr@5', 
        index='chunk_size', 
        columns='model', 
        aggfunc='mean'
    )
    # Raccourcir les noms de colonnes
    pivot.columns = [c.split('/')[-1][:20] for c in pivot.columns]
    
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax, 
                linewidths=0.5, cbar_kws={'label': 'MRR@5'})
    ax.set_title('Interaction chunk_size × modèle d\'embedding (MRR@5 moyen)', fontweight='bold')
    ax.set_ylabel('Taille de chunk (tokens)')
    ax.set_xlabel('Modèle d\'embedding')
    plt.tight_layout()
    plt.savefig(BENCHMARK_DIR / 'heatmap_chunk_model.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n💡 Cette heatmap montre les interactions entre dimensions —")
    print("   c'est pourquoi le grid search complet est nécessaire (vs. OFAT).")

---

## 📦 Étape 4 — Indexation vectorielle (FAISS)

### Objectif
Construire l'index de recherche final en appliquant la **configuration optimale** identifiée par le benchmarking (étape 3).

### Processus
1. **Lecture** de la configuration optimale depuis `benchmark_report.json`
2. **Découpage** de l'ensemble du corpus avec les paramètres de chunk optimaux
3. **Vectorisation** des chunks avec le modèle d'embedding optimal
4. **Construction** d'un index FAISS (`IndexFlatIP` — recherche exacte par produit scalaire)
5. **Sauvegarde** de l'index et des métadonnées

### Sortie attendue
- `data/processed/vectorstore/faiss_index.bin` — index binaire FAISS
- `data/processed/vectorstore/chunks_metadata.json` — métadonnées des chunks

### Référence
- Johnson, J. et al. (2019). *Billion-scale similarity search with GPUs*. IEEE TBBDATA.

> ⏱️ **Durée estimée** : 10–30 minutes

In [ ]:
# ============================================================
# ÉTAPE 4 — INDEXATION VECTORIELLE (FAISS)
# ============================================================
from etape4_indexation import main as run_etape4

run_etape4()

In [ ]:
# Vérification de l'index
import faiss
from config import VECTORSTORE_DIR

index_path = VECTORSTORE_DIR / "faiss_index.bin"
chunks_path = VECTORSTORE_DIR / "chunks_metadata.json"

if index_path.exists():
    index = faiss.read_index(str(index_path))
    with open(chunks_path, 'r', encoding='utf-8') as f:
        chunks_meta = json.load(f)
    
    print(f"✅ Index FAISS chargé :")
    print(f"   Dimension     : {index.d}")
    print(f"   Nb. vecteurs  : {index.ntotal}")
    print(f"   Nb. chunks    : {len(chunks_meta)}")
    
    # Distribution par source
    sources = {}
    for c in chunks_meta:
        src = c.get('doc_source', 'inconnu')
        sources[src] = sources.get(src, 0) + 1
    print(f"\n   Distribution par source :")
    for src, count in sorted(sources.items()):
        print(f"     {src:15s} : {count:>5} chunks")
else:
    print("⚠️ Index non trouvé. Exécutez l'étape 4.")

---

## 🤖 Étape 5 — Recherche et Génération RAG

### Objectif
Implémenter le cœur opérationnel du système RAG : étant donné une question en langage naturel, **retrouver** les passages pertinents dans l'index puis **générer** une réponse synthétique et sourcée via un LLM.

### Architecture du pipeline RAG

```
Question utilisateur
       │
       ▼
┌─────────────────┐
│  Embedding       │  ← Modèle optimal (étape 3)
│  de la requête   │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Recherche       │  ← FAISS + BM25 (hybride)
│  top-k passages  │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Construction    │  ← Prompt structuré avec
│  du prompt       │    passages + instructions
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Génération      │  ← Mistral 7B Instruct
│  de la réponse   │    (4-bit quantized)
└────────┬────────┘
         │
         ▼
   Réponse + Sources
```

### Modèle de langue : Mistral 7B Instruct

| Caractéristique | Valeur |
|---|---|
| Paramètres | 7,3 milliards |
| Architecture | Transformer (décodeur), GQA, Sliding Window Attention |
| Quantification | 4-bit NF4 via `bitsandbytes` (~5 Go VRAM) |
| Fenêtre de contexte | 8 192 tokens |
| Licence | Apache 2.0 |
| Référence | Jiang et al. (2023) |

> ⏱️ **Durée estimée** : 3–5 minutes (chargement du modèle) + ~10s par question

In [ ]:
# ============================================================
# ÉTAPE 5 — CHARGEMENT DU PIPELINE RAG
# ============================================================
from etape5_generation import (
    load_index, load_search_config, auto_detect_client, RAGPipeline
)
from sentence_transformers import SentenceTransformer

# 1. Charger l'index FAISS et les chunks
index, chunks = load_index()
print(f"✅ {len(chunks)} chunks chargés, dimension={index.d}")

# 2. Charger la configuration optimale
search_config = load_search_config()
model_name = search_config.get('embedding_model', 'all-MiniLM-L6-v2')
print(f"✅ Config : embedding={model_name}, search={search_config['search_method']}")

# 3. Charger le modèle d'embedding
embedding_model = SentenceTransformer(model_name)

# 4. Charger le LLM (auto-détection GPU/CPU)
llm_client = auto_detect_client()

# 5. Créer le pipeline RAG
pipeline = RAGPipeline(
    llm_client=llm_client,
    index=index,
    chunks=chunks,
    embedding_model=embedding_model,
    search_config=search_config,
)
print("\n🤖 Pipeline RAG initialisé et prêt.")

In [ ]:
# ============================================================
# DÉMONSTRATION : Questions sur Python
# ============================================================
python_questions = [
    "What is a Python decorator and how to use it?",
    "How does Python's garbage collector handle circular references?",
    "What is the difference between a list and a tuple in Python?",
]

print("=" * 70)
print("🐍  DÉMONSTRATION — Questions Python")
print("=" * 70)

for q in python_questions:
    print(f"\n❓ {q}")
    result = pipeline.answer(q)
    print(f"\n💬 {result['answer'][:500]}")
    sources = set(s.get('doc_title', '?') for s in result['sources'])
    print(f"\n📚 Sources : {', '.join(list(sources)[:3])}")
    print("─" * 70)

In [ ]:
# ============================================================
# DÉMONSTRATION : Questions sur Scikit-learn
# ============================================================
sklearn_questions = [
    "How to perform cross-validation with Scikit-learn?",
    "What is the difference between fit() and fit_transform()?",
    "How to handle missing values in Scikit-learn?",
]

print("=" * 70)
print("🔬  DÉMONSTRATION — Questions Scikit-learn")
print("=" * 70)

for q in sklearn_questions:
    print(f"\n❓ {q}")
    result = pipeline.answer(q)
    print(f"\n💬 {result['answer'][:500]}")
    sources = set(s.get('doc_title', '?') for s in result['sources'])
    print(f"\n📚 Sources : {', '.join(list(sources)[:3])}")
    print("─" * 70)

In [ ]:
# ============================================================
# DÉMONSTRATION : Questions sur LangChain
# ============================================================
langchain_questions = [
    "How to create a custom retriever in LangChain?",
    "What is LCEL (LangChain Expression Language)?",
    "How to stream responses from an LLM in LangChain?",
]

print("=" * 70)
print("🦜  DÉMONSTRATION — Questions LangChain")
print("=" * 70)

for q in langchain_questions:
    print(f"\n❓ {q}")
    result = pipeline.answer(q)
    print(f"\n💬 {result['answer'][:500]}")
    sources = set(s.get('doc_title', '?') for s in result['sources'])
    print(f"\n📚 Sources : {', '.join(list(sources)[:3])}")
    print("─" * 70)

---

## 📈 Étape 6 — Évaluation RAGAS du système

### Objectif
Évaluer quantitativement la qualité du système RAG complet (recherche + génération) selon le cadre **RAGAS** (Es et al., 2024).

### Métriques RAGAS

| Métrique | Ce qu'elle mesure | Approche |
|---|---|---|
| **Faithfulness** | Les affirmations de la réponse sont-elles soutenues par les passages retrouvés ? | Extraction de claims → vérification par LLM |
| **Answer Relevancy** | La réponse est-elle pertinente par rapport à la question ? | Génération de questions inverses → similarité cosinus |
| **Context Precision** | Les passages retrouvés sont-ils pertinents ? | Jugement LLM passage par passage |
| **Context Recall** | Les passages couvrent-ils les infos nécessaires ? | Décomposition réponse de référence → vérification |

### Jeu d'évaluation
20 questions avec **réponses de référence** (ground truth) :
- 7 questions Python (décorateurs, GIL, gestion mémoire, etc.)
- 6 questions Scikit-learn (GridSearch, pipeline, K-Fold, etc.)
- 7 questions LangChain (agents, mémoire, LCEL, etc.)

### Référence
- Es, S. et al. (2024). *RAGAS: Automated Evaluation of Retrieval Augmented Generation*.

> ⏱️ **Durée estimée** : 30–60 minutes (20 questions × 4 métriques × appels LLM)

In [ ]:
# ============================================================
# ÉTAPE 6 — ÉVALUATION RAGAS
# ============================================================
from etape6_evaluation import main as run_etape6

run_etape6()

In [ ]:
# ============================================================
# Analyse détaillée des résultats RAGAS
# ============================================================
from config import EVALUATION_DIR

report_path = EVALUATION_DIR / "ragas_report.json"

if report_path.exists():
    with open(report_path, 'r', encoding='utf-8') as f:
        ragas_report = json.load(f)
    
    print("=" * 60)
    print("📊  RÉSULTATS RAGAS — MOYENNES GLOBALES")
    print("=" * 60)
    
    metrics = ragas_report['mean_metrics']
    for name, score in metrics.items():
        bar = '█' * int(score * 40) + '░' * (40 - int(score * 40))
        print(f"  {name:22s} : {bar} {score:.4f}")
    
    # Score global RAGAS (moyenne harmonique)
    import numpy as np
    values = list(metrics.values())
    harmonic_mean = len(values) / sum(1.0/v if v > 0 else 100 for v in values)
    print(f"\n  {'Score RAGAS global':22s} : {harmonic_mean:.4f} (moyenne harmonique)")
    
else:
    print("⚠️ Rapport RAGAS non trouvé. Exécutez l'étape 6.")

In [ ]:
# ============================================================
# Visualisation des résultats RAGAS
# ============================================================
if report_path.exists():
    details_path = EVALUATION_DIR / "ragas_details.csv"
    if details_path.exists():
        df_ragas = pd.read_csv(details_path)
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle('Évaluation RAGAS du système RAG', fontsize=14, fontweight='bold')
        
        # 1. Radar chart des métriques globales
        ax1 = axes[0]
        categories = list(metrics.keys())
        values_plot = list(metrics.values()) + [list(metrics.values())[0]]  # Fermer le polygone
        angles = [n / float(len(categories)) * 2 * 3.14159 for n in range(len(categories))]
        angles += angles[:1]
        
        ax1 = fig.add_subplot(121, polar=True)
        ax1.plot(angles, values_plot, 'o-', linewidth=2, color='#2196F3')
        ax1.fill(angles, values_plot, alpha=0.25, color='#2196F3')
        ax1.set_xticks(angles[:-1])
        ax1.set_xticklabels([c.replace('_', '\n') for c in categories], size=9)
        ax1.set_ylim(0, 1)
        ax1.set_title('Métriques RAGAS globales', pad=20)
        
        # 2. Scores par source
        ax2 = fig.add_subplot(122)
        metric_cols = ['Faithfulness', 'Answer_Relevancy', 'Context_Precision', 'Context_Recall']
        available_cols = [c for c in metric_cols if c in df_ragas.columns]
        
        if 'Source' in df_ragas.columns and available_cols:
            for col in available_cols:
                df_ragas[col] = pd.to_numeric(df_ragas[col], errors='coerce')
            source_means = df_ragas.groupby('Source')[available_cols].mean()
            source_means.plot(kind='bar', ax=ax2, alpha=0.8)
            ax2.set_title('Scores RAGAS par source documentaire')
            ax2.set_ylabel('Score')
            ax2.set_ylim(0, 1)
            ax2.legend(fontsize=8, loc='lower right')
            ax2.tick_params(axis='x', rotation=0)
        
        plt.tight_layout()
        plt.savefig(EVALUATION_DIR / 'ragas_visualisation.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"\n📊 Graphique sauvegardé : {EVALUATION_DIR / 'ragas_visualisation.png'}")

---

## 📋 Synthèse des résultats

Cette section résume les résultats obtenus à travers les 6 étapes du pipeline.

In [ ]:
# ============================================================
# SYNTHÈSE COMPLÈTE DU PIPELINE
# ============================================================

print("╔" + "═" * 68 + "╗")
print("║" + " SYNTHÈSE DU PIPELINE RAG ".center(68) + "║")
print("╠" + "═" * 68 + "╣")

# Étape 1-2 : Corpus
total_clean = 0
for src in ['python', 'sklearn', 'langchain']:
    src_dir = CLEAN_DIR / src
    if src_dir.exists():
        total_clean += len(list(src_dir.glob('*.json')))
print(f"║  📥 Corpus nettoyé         : {total_clean:>6} documents" + " " * (68-48) + "║")

# Étape 3 : Benchmarking
if grid_path.exists():
    df_grid = pd.read_csv(grid_path)
    best_grid = df_grid.loc[df_grid['mrr@5'].idxmax()]
    print(f"║  📊 Configurations testées : {len(df_grid):>6}" + " " * (68-41) + "║")
    print(f"║  🏆 Meilleur MRR@5        : {best_grid['mrr@5']:>6.4f}" + " " * (68-43) + "║")

# Étape 4 : Index
if index_path.exists():
    idx = faiss.read_index(str(index_path))
    print(f"║  📦 Chunks indexés         : {idx.ntotal:>6}" + " " * (68-41) + "║")
    print(f"║  📐 Dimension embedding    : {idx.d:>6}" + " " * (68-41) + "║")

# Étape 5 : LLM
print(f"║  🤖 Modèle LLM            : Mistral 7B Instruct (4-bit)" + " " * (68-56) + "║")

# Étape 6 : RAGAS
if report_path.exists():
    for name, score in metrics.items():
        label = f"  📈 {name:22s} : {score:.4f}"
        padding = 68 - len(label)
        print(f"║{label}" + " " * max(1, padding) + "║")

print("╚" + "═" * 68 + "╝")

---

## 🔬 Bonus : Mode interactif

Utilisez la cellule ci-dessous pour poser vos propres questions au système RAG.

In [ ]:
# ============================================================
# MODE INTERACTIF — Posez vos questions !
# ============================================================
# Modifiez la variable `question` et exécutez cette cellule.

question = "How does Python's asyncio event loop work?"  # ← Changez cette question

print(f"❓ Question : {question}\n")
result = pipeline.answer(question)

print("💬 Réponse :")
print("─" * 60)
print(result['answer'])
print("─" * 60)

print("\n📚 Sources utilisées :")
seen = set()
for src in result['sources']:
    title = src.get('doc_title', 'Inconnu')
    source = src.get('doc_source', '?')
    key = f"{title} ({source})"
    if key not in seen:
        print(f"  • {key}")
        seen.add(key)

print(f"\n📊 Scores de retrieval : {[f'{s:.3f}' for s in result['retrieval_scores']]}")

---

## 📝 Conclusion

Ce notebook a présenté l'exécution complète du pipeline RAG en 6 étapes :

1. **Collecte** de la documentation technique (Python, Sklearn, LangChain) par clonage Git
2. **Nettoyage** et normalisation des documents (RST, MDX → JSON)
3. **Benchmarking** exhaustif de 108 configurations (grid search) pour identifier les paramètres optimaux
4. **Indexation** vectorielle FAISS avec la configuration optimale
5. **Génération** de réponses via Mistral 7B Instruct (quantifié 4-bit)
6. **Évaluation** RAGAS multi-dimensionnelle (Faithfulness, Relevancy, Context Precision/Recall)

### Points forts du système
- Architecture modulaire et reproductible
- Choix techniques justifiés par l'expérimentation (benchmarking data-driven)
- Évaluation conforme à l'état de l'art (RAGAS)

### Limites et perspectives
- Jeu d'évaluation limité à 20 questions (extensible)
- Pas de *query rewriting* ni de *re-ranking* (Advanced RAG)
- Évaluation humaine non réalisée

---

*Pipeline développé dans le cadre d'un stage de fin d'études à l'INSEA, encadré par 3D Smart Factory.*